# Notebook C — controlled legacy-development workflow

Historical saved outputs are preserved verbatim and predate this edited source. They are not a new execution or corrected research evidence. Notebook B archival outputs identify f538726; the readiness base is f66bb57.

Run in a fresh kernel. Explicitly acknowledge legacy data/checkpoint semantics. Use a fresh OUTPUT_DIR and the identical code, data, checkpoint, units and split in A–D. A → B → C/D; C requires A and B, D requires B. Scientific projected training uses the separately documented audited baseline. No CAMELS transfer claims from legacy GroupMass.


In [ ]:
# Settings — edit these before Run All.
import os
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/cosmic-net")
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "codex/fix-stageb-budget-feasibility"
SYNC_REPO = True                 # False for an already prepared local checkout.
INSTALL_MISSING = True
INPUT_DIR = None                 # None: discover one matching Kaggle dataset.
INPUT_ROOT = Path("/kaggle/input")
DEVICE = "auto"                  # "auto", "cpu", or "cuda"
SMOKE_TEST = False
SMOKE_GRAPHS_PER_SPLIT = 3
GRAPH_BACKEND = "auto"           # "auto" or "torch" (no native extensions)
OUTPUT_DIR = None  # Default: outputs/rls/notebook_C
B_ARTIFACT_DIR = None  # Folder containing B's policy.pt AND provenance.json.
RUN_CAMELS = False
CAMELS_HDF5 = None  # Attach a real z=0 group catalog to enable OOD evaluation.

A_ARTIFACT_DIR = None  # Folder containing A's baselines.csv AND provenance.json.
EVALUATE_STAGE_B = False  # Load B's fine-tuned model as an additional comparison.
COVERAGE_SAMPLES = 30

ACKNOWLEDGE_LEGACY_INPUTS = False  # Required for a non-smoke historical CSV/checkpoint campaign.
EXPECTED_REPO_COMMIT = None       # Set to the exact verified commit before the campaign.
LEGACY_CSV_UNITS = {"position": "ckpc_per_h", "radius": "ckpc_per_h", "stellar_mass": "log10_Msun", "halo_mass": "M_sun"}
FRACTIONS = [0.4, 0.6, 0.8, 1.0]

# Explicit settings for this new private campaign.
EXPECTED_REPO_COMMIT = 'c825bf57652005c80faff79cf4dfd58e3a9baa14'
ACKNOWLEDGE_LEGACY_INPUTS = True
SYNC_REPO = False
DEVICE = "cuda"
INPUT_ROOT = Path("/kaggle/input")
_input_candidates = [Path("/kaggle/input/datasets/nealsalian/cosmicnet-data"),
                     Path("/kaggle/input/cosmicnet-data")]
INPUT_DIR = next((path for path in _input_candidates if (path / "tng100_clustered.csv").is_file()), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Pinned cosmicnet-data inputs are not mounted")
OUTPUT_DIR = Path("/kaggle/working/cosmic-net-runs/notebook_C")


In [ ]:
# Dependencies — retain Kaggle's PyTorch and install only missing core packages.
import importlib
import importlib.util
import subprocess
import torch

print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| CUDA build:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
packages = {"torch_geometric": "torch-geometric>=2.6,<3", "yaml": "pyyaml",
            "numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
            "matplotlib": "matplotlib", "h5py": "h5py", "requests": "requests", "dotenv": "python-dotenv"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError("Missing packages: " + ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    importlib.invalidate_caches()
# NNConv works without torch-scatter/torch-sparse. Do not install CPU extension
# wheels into a CUDA runtime or compile unsupported wheels during notebook setup.
import torch_geometric
print("PyG:", torch_geometric.__version__)


In [ ]:
# Verified local Git bundle delivery; no GitHub branch push.
import base64, hashlib, subprocess
_delivery = base64.b64decode('...')
assert hashlib.sha256(_delivery).hexdigest() == '88043f8cfcf484907989a6c78608add7b70a2c576a974870b4b619e72f08c47b'
_bundle_path = Path("/kaggle/working/cosmic-net-stageb-readiness.bundle")
_bundle_path.write_bytes(_delivery)
_git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
if REPO_DIR.exists():
    raise FileExistsError("Use a fresh kernel; refusing to overwrite an existing checkout")
subprocess.run(["git", "clone", "--depth", "1", "--no-checkout", "--single-branch",
    "--branch", "fix/research-pipeline-readiness", REPO_URL, str(REPO_DIR)], check=True, env=_git_env)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--no-auto-gc", str(_bundle_path),
    "refs/heads/codex/fix-stageb-budget-feasibility"], check=True, env=_git_env)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", 'c825bf57652005c80faff79cf4dfd58e3a9baa14'], check=True, env=_git_env)
REPO_HEAD = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
assert REPO_HEAD == EXPECTED_REPO_COMMIT == 'c825bf57652005c80faff79cf4dfd58e3a9baa14'
sys.path.insert(0, str(REPO_DIR))
print("Verified evaluator commit:", REPO_HEAD, "| delivery: private local Git bundle")


In [ ]:
# Locate and validate the attached dataset before any model or graph work.
import numpy as np
import pandas as pd

def locate_inputs(input_dir, input_root):
    if input_dir is not None:
        candidates = [Path(input_dir)]
    else:
        candidates = sorted({p.parent for p in Path(input_root).rglob("tng100_clustered.csv")
                             if (p.parent / "best_model_augmented.pt").is_file()})
    if len(candidates) != 1:
        raise FileNotFoundError("Set INPUT_DIR to the dataset folder containing "
                                "tng100_clustered.csv and best_model_augmented.pt. "
                                f"Matching folders: {candidates}")
    folder = candidates[0].resolve()
    csv_path = folder / "tng100_clustered.csv"
    checkpoint_path = folder / "best_model_augmented.pt"
    for path in (csv_path, checkpoint_path):
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing or empty input: {path}")
        with path.open("rb") as stream:
            if stream.read(128).startswith(b"version https://git-lfs.github.com/spec/v1"):
                raise ValueError(f"{path} is a Git LFS pointer, not the actual data file.")
    return csv_path, checkpoint_path

CSV_PATH, CHECKPOINT_PATH = locate_inputs(INPUT_DIR, INPUT_ROOT)
catalog = pd.read_csv(CSV_PATH)
velocity_column = "vel_dispersion" if "vel_dispersion" in catalog else "velocity_dispersion"
required = ["subhalo_id", "group_id", "stellar_mass", velocity_column,
            "half_mass_radius", "metallicity", "pos_x", "pos_y", "pos_z",
            "vel_x", "vel_y", "vel_z"]
target_column = "halo_mass" if "halo_mass" in catalog else "halo_mass_log"
required.append(target_column)
missing_columns = sorted(set(required) - set(catalog.columns))
if missing_columns:
    raise ValueError(f"CSV missing required columns: {missing_columns}")
numeric = catalog[required].apply(pd.to_numeric, errors="coerce")
if catalog.empty or not np.isfinite(numeric.to_numpy()).all():
    raise ValueError("CSV must contain finite numeric features and targets in every row.")
if (numeric[[velocity_column, "half_mass_radius", "metallicity"]] < 0).any().any():
    raise ValueError("Velocity dispersion, radius and metallicity cannot be negative.")
print("CSV:", CSV_PATH, "| rows:", len(catalog), "| halos:", catalog.group_id.nunique())
print("Checkpoint:", CHECKPOINT_PATH, "| bytes:", CHECKPOINT_PATH.stat().st_size)


In [ ]:
# Configuration and imports — retain the repository's RL settings.
import copy
import yaml
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder
from model.model import build_model
from model.physics_loss import MetricsComputer
from rls.provenance import record_backbone, require_backbone_label

_C_IMPORTED_HEAD = REPO_HEAD
cfg = yaml.safe_load((REPO_DIR / "config/config.yaml").read_text())
cfg["data"].update(source="tng", num_workers=0, batch_size=16)
cfg["data"]["tng"]["clustered_file"] = str(CSV_PATH)
cfg["data"]["tng"]["clustered_units"] = copy.deepcopy(LEGACY_CSV_UNITS)
cfg["data"]["checkpoint_research_label"] = "legacy_total_radius"
cfg["rls"].update(evaluation_decoder="direct", evaluation_constraint="no_isolates", edge_feature_names=cfg["graph"]["edge_features"])
if not SMOKE_TEST and not ACKNOWLEDGE_LEGACY_INPUTS:
    raise ValueError("Explicit ACKNOWLEDGE_LEGACY_INPUTS=True is required; these inputs do not certify corrected astronomy research.")
if DEVICE not in ("auto", "cpu", "cuda"):
    raise ValueError("DEVICE must be auto, cpu or cuda.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable. Enable a Kaggle GPU or use DEVICE='cpu'.")
device = torch.device("cuda" if DEVICE == "cuda" or
                      (DEVICE == "auto" and torch.cuda.is_available()) else "cpu")
if GRAPH_BACKEND not in ("auto", "torch"):
    raise ValueError("GRAPH_BACKEND must be auto or torch.")
if SMOKE_GRAPHS_PER_SPLIT < 2:
    raise ValueError("Use at least two smoke graphs per split.")
from rls.notebook_workflow import fresh_output_directory
OUT = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else Path("/kaggle/working/cosmic-net-runs/notebook_C")
if SMOKE_TEST:
    OUT = OUT / "smoke"
OUT = fresh_output_directory(OUT)
cfg["data"]["tng"]["cache_dir"] = str(OUT / "tng_cache")
import json
(OUT / "provenance.json").write_text(json.dumps({"C": {"stage": "C", "status": "running", "repo": {"head": REPO_HEAD}}}, indent=2))
seed = int(cfg.get("seed", 42))
torch.manual_seed(seed)
np.random.seed(seed)
print("Device:", device, "| run mode:", "SMOKE" if SMOKE_TEST else "FULL", "| outputs:", OUT)


In [ ]:
# Load only the supplied checkpoint; use its own architecture and graph settings.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
checkpoint_config = checkpoint.get("config") if isinstance(checkpoint, dict) else None
if checkpoint_config and "model" in checkpoint_config:
    cfg["model"] = copy.deepcopy(checkpoint_config["model"])
    if "graph" in checkpoint_config:
        cfg["graph"] = copy.deepcopy(checkpoint_config["graph"])
cfg["model"]["mc_samples"] = 30
if cfg["graph"].get("hierarchical", False):
    raise ValueError("Notebook C supports single-level graphs; hierarchical mode is only a scaffold.")
from model.model import load_model
# Legacy metadata never upgrades these weights to projected/audited artifacts.
if checkpoint.get("artifact_type") == "projected_backbone":
    raise ValueError("Use the audited projected workflow; this notebook expects the explicit legacy schema.")
gnn = load_model(str(CHECKPOINT_PATH), cfg, device, research_label="legacy_total_radius")
gnn = gnn.to(device).eval()
gnn.requires_grad_(False)  # Saliency still differentiates input features.
model_device = next(gnn.parameters()).device
backbone_record = record_backbone("frozen", gnn)
del checkpoint
print("Loaded:", CHECKPOINT_PATH.name, "| dimensions:", gnn.hidden_dim, gnn.output_dim,
      "| parameters:", sum(p.numel() for p in gnn.parameters()))

cfg["rls"]["edge_feature_names"] = list(cfg["graph"]["edge_features"])


In [ ]:
# Build each halo independently; fail on invalid graphs instead of dropping them.
def validate_graph(graph):
    if graph.num_nodes < 1 or graph.edge_index.shape[1] == 0:
        raise ValueError(f"Empty graph or no edges: {graph.cluster_id}")
    graph.validate(raise_on_error=True)
    if graph.x.shape[1] != gnn.node_input_dim or graph.edge_attr.shape != (
            graph.edge_index.shape[1], gnn.edge_input_dim):
        raise ValueError(f"Feature dimensions incompatible with checkpoint: {graph.cluster_id}")
    for name in ("x", "pos", "edge_attr", "y"):
        value = getattr(graph, name, None)
        if value is None or not torch.isfinite(value).all():
            raise ValueError(f"Missing or non-finite {name}: {graph.cluster_id}")
    if graph.pos.shape != (graph.num_nodes, 3):
        raise ValueError(f"Expected physical 3D positions: {graph.cluster_id}")

loader = get_loader(cfg)
halos = loader.load()
split_halos = dict(zip(("train", "val", "test"), loader.split_data(halos)))
full_split_sizes = {name: len(items) for name, items in split_halos.items()}
if any(size < 2 for size in full_split_sizes.values()):
    raise ValueError(f"Each split needs at least two halos; got {full_split_sizes}.")
if SMOKE_TEST:
    split_halos = {name: items[:SMOKE_GRAPHS_PER_SPLIT] for name, items in split_halos.items()}
torch.manual_seed(seed)
np.random.seed(seed)
builder = GraphBuilder(cfg)
builder.backend_used = "portable_torch"
builder.backend_used = "portable_torch"
import time
graph_start = time.perf_counter()
import time
graph_start = time.perf_counter()
split_graphs = {}
for name, items in split_halos.items():
    graphs = []
    for halo in items:
        graph = builder.build_graph(halo)
        validate_graph(graph)
        graphs.append(graph)
    split_graphs[name] = graphs
train_loader = DataLoader(split_graphs["train"], batch_size=cfg["data"]["batch_size"], shuffle=True)
val_loader = DataLoader(split_graphs["val"], batch_size=cfg["data"]["batch_size"], shuffle=False)
test_loader = DataLoader(split_graphs["test"], batch_size=cfg["data"]["batch_size"], shuffle=False)
print("Full split:", full_split_sizes, "| evaluated split:",
      {name: len(graphs) for name, graphs in split_graphs.items()},
      "| graph backend: portable torch | tie policy: inclusive")
graph_construction_time_s = time.perf_counter() - graph_start

graph_construction_time_s = time.perf_counter() - graph_start


In [ ]:
from rls.notebook_workflow import (
    run_context, file_info, checked_artifact, source_artifacts, load_saved_policy,
    prepared_splits, predict_graph, prediction_pair, policy_masks, evaluate_masks,
    result_rows, merge_baselines, physics_diagnostics, coverage_rows,
    adaptation_trial, validation_gates, complete_stage, local_camels_graphs,
    policy_metrics, random_pair_reference, policy_validation)
from rls.evaluate import save_paper_plots
from rls.policy import build_policy
from rls.policy_gradient import ValueNet, PolicyGradientTrainer
from rls.train_policy import train_policy
from rls.stageb import fine_tune_gnn
context = run_context(cfg, CSV_PATH, CHECKPOINT_PATH, split_graphs, backbone_record, SMOKE_TEST)
prepared = prepared_splits(split_graphs, gnn, device)
train_graphs, val_graphs, test_graphs = (prepared[name] for name in ("train", "val", "test"))
artifacts = []
dependencies = {}
run_mode = context["run_mode"]
g = test_graphs[0]
full_mask = torch.ones(g["edge_index"].shape[1], dtype=torch.bool, device=device)
print("=== DEVICE CHECK ===")
print("GNN:", next(gnn.parameters()).device)
for key in ("x", "edge_index", "edge_attr", "emb", "ctx"):
    print(key + ":", g[key].device)
print("Single-halo full-graph prediction:", float(predict_graph(gnn, g, full_mask)))

context["graph_construction_time_s"] = graph_construction_time_s

from rls.notebook_budget import budget_cohort, require_budget_cohort, record_budget_realizations


In [ ]:
B_DIR, b_manifest = source_artifacts(B_ARTIFACT_DIR, INPUT_ROOT, "B", "policy.pt", context)
policy = load_saved_policy(B_DIR, b_manifest, cfg, gnn, device)
dependencies["B"] = file_info(B_DIR / "provenance.json")
print("Loaded policy:", B_DIR / "policy.pt", "| decoder:", cfg["rls"]["sparsity_mode"])


In [ ]:
A_DIR, a_manifest = source_artifacts(A_ARTIFACT_DIR, INPUT_ROOT, "A", "baselines.csv", context)
dependencies["A"] = file_info(A_DIR / "provenance.json")


In [ ]:
from rls.notebook_benchmark import matched_notebook_rows, load_learned_control
learned_path = checked_artifact(A_DIR, a_manifest, "learned_control.pt")
learned, learned_artifact = load_learned_control(learned_path, context, gnn)
# Re-evaluate every method together; never concatenate incompatible A rows.
context["config_used"] = copy.deepcopy(cfg)
rows, comparison_per_example = matched_notebook_rows(gnn, split_graphs["test"], context,
    a_manifest["fractions"], policy=policy, learned=learned, learned_artifact=learned_artifact)
results_df = save_paper_plots(rows, str(OUT))
(OUT / "matched_per_example.json").write_text(json.dumps(comparison_per_example, indent=2, allow_nan=False))
artifacts += ["results_table.csv", "pareto_and_fidelity.png", "matched_per_example.json"]
# Keep original benchmark failures above; diagnostics use an aligned feasible cohort.
diagnostic_graphs, diagnostic_budget_report = budget_cohort(test_graphs, cfg["rls"],
    split="test", target_sparsity=cfg["rls"]["target_sparsity_end"])
(OUT / "budget_eligibility.json").write_text(json.dumps(diagnostic_budget_report, indent=2, allow_nan=False))
artifacts.append("budget_eligibility.json")
test_masks, test_probabilities = policy_masks(policy, diagnostic_graphs, cfg["rls"],
    target_sparsity=cfg["rls"]["target_sparsity_end"])
if EVALUATE_STAGE_B:
    print("Stage B remains a separately labelled exploratory model; primary comparisons use the frozen backbone.")
print(results_df[["method", "frac", "rmse", "mae", "r2", "n_failed", "common_n"]].to_string(index=False))

record_budget_realizations(diagnostic_budget_report, diagnostic_graphs, test_masks)
(OUT / "budget_eligibility.json").write_text(json.dumps(diagnostic_budget_report, indent=2, allow_nan=False))


In [ ]:
physics_df = pd.DataFrame(physics_diagnostics(diagnostic_graphs, test_probabilities, test_masks),
    columns=["cluster_id", "spearman_rho", "binding_mannwhitney_p", "physical_edges", "note"])
physics_df.to_csv(OUT / "physics_alignment.csv", index=False)
artifacts.append("physics_alignment.csv")
print(physics_df.to_string(index=False))


In [ ]:
coverage_samples = min(COVERAGE_SAMPLES, 3) if SMOKE_TEST else COVERAGE_SAMPLES
coverage_df = pd.DataFrame(
    coverage_rows(gnn, diagnostic_graphs, test_masks, coverage_samples) if diagnostic_graphs else [],
    columns=["method", "coverage_95", "mean_interval_width", "mc_samples"])
if not diagnostic_graphs:
    print("Coverage skipped: no exact-budget feasible graphs; see budget_eligibility.json.")
coverage_df["run_mode"] = run_mode
coverage_df.to_csv(OUT / "calibration.csv", index=False)
artifacts.append("calibration.csv")
print(coverage_df.to_string(index=False))


In [ ]:
if RUN_CAMELS:
    raise ValueError("Legacy CAMELS GroupMass is incompatible with the uncertified TNG target. Use audited manifest/projected artifacts and rls.astronomy_evaluation; no legacy OOD claim is produced.")
print("CAMELS disabled: audited compatible targets and projected research artifacts are required.")


In [ ]:
assert record_backbone("frozen", gnn) == backbone_record, "Frozen backbone changed during this run."
context["config_used"] = copy.deepcopy(cfg)
manifest = complete_stage(OUT, "C", context, artifacts,
    repo={"head": REPO_HEAD, "branch": REPO_BRANCH}, graph_backend=builder.backend_used,
    full_split_sizes=full_split_sizes, device=str(device), torch=str(torch.__version__),
    dependencies=dependencies)
